# 1. Operator identities and joint-measurability thresholds (Table 1)

This notebook reproduces

* the machine-precision checks of the joint POVM stated at the start of Section 6
  (positivity, completeness, marginals, Lemma on anticommuting observables, Born rule of the hard instances), and
* **Table 1**: joint-measurability thresholds $\eta^\star$ obtained by semidefinite programming,
  compared with the closed forms $1/\sqrt G$ (Proposition 2) and $(\cos\frac\theta2+\sin\frac\theta2)^{-1}$ (Proposition 4),
  and the incompatibility numbers $\kappa = 1/\eta^{\star2}$.

Outputs: `results/sdp_eta_main.json` (also used by notebook 3 for Fig. 4a) and `results/table1_sdp.tex`.
Run time: a few seconds.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULTS, FIGURES, REFERENCE = ROOT / "results", ROOT / "figures", ROOT / "reference"
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)

from copycost.operators import verify_block
from copycost.sdp import eta_star, pair_observables, pair_threshold
from copycost.operators import anticommuting_set

## Exact operator checks

For each block size $G$ the first $G$ Jordan–Wigner operators on $q$ qubits are used. `verify_block` raises an
`AssertionError` if any check fails; the table reports the worst deviations. The last-but-one column is
$\max \sum_i \langle A_i\rangle^2$ over 2000 random pure states, which must not exceed 1.

In [2]:
rng = np.random.default_rng(1)
checks = pd.DataFrame([verify_block(G, q, rng) for G, q in [(2, 1), (3, 1), (4, 2), (5, 2), (7, 3), (9, 4)]])
checks.style.format({c: "{:.1e}" for c in ["min_eigenvalue", "completeness_err", "marginal_err", "born_rule_err"]}
                    | {"max_sum_sq_expectations": "{:.4f}"})

,G,qubits,dim,min_eigenvalue,completeness_err,marginal_err,max_sum_sq_expectations,born_rule_err
0,2,1,2,0.0e+00,0.0e+00,0.0e+00,1.0000,0.0e+00
1,3,1,2,-2.0e-17,2.8e-17,0.0e+00,1.0000,1.4e-17
2,4,2,4,-3.1e-33,0.0e+00,0.0e+00,1.0000,0.0e+00
3,5,2,4,-1.7e-17,1.5e-17,1.1e-16,1.0000,0.0e+00
4,7,3,8,-4.7e-18,8.6e-18,4.4e-16,0.9999,8.7e-19
5,9,4,16,-2.0e-18,1.4e-17,1.0e-15,0.9531,2.2e-19


## Semidefinite programs

$\eta^\star = \max \eta$ such that a POVM $\{F_b\}_{b\in\{\pm1\}^G}$ exists with marginals $(I+\eta A_k)/2$.
Pairs: $A = Z$, $B = \cos\theta\,Z + \sin\theta\,X$ for $\theta = 0^\circ, 10^\circ, \dots, 90^\circ$.
Blocks: the first $G$ anticommuting operators, $G = 1,\dots,5$.

In [3]:
sdp = {"pair": {"theta_deg": [], "eta_sdp": []}, "block": {"G": [], "eta_sdp": []}}
for th in np.linspace(0, 90, 10):
    sdp["pair"]["theta_deg"].append(float(th))
    sdp["pair"]["eta_sdp"].append(eta_star(pair_observables(th)))
for G, q in [(1, 1), (2, 1), (3, 1), (4, 2), (5, 2)]:
    sdp["block"]["G"].append(G)
    sdp["block"]["eta_sdp"].append(eta_star(anticommuting_set(q)[:G]))
(RESULTS / "sdp_eta_main.json").write_text(json.dumps(sdp, indent=1))

pair = pd.DataFrame({"theta_deg": sdp["pair"]["theta_deg"], "SDP eta*": sdp["pair"]["eta_sdp"]})
pair["closed form"] = pair_threshold(pair["theta_deg"])
pair["|difference|"] = (pair["SDP eta*"] - pair["closed form"]).abs()
block = pd.DataFrame({"G": sdp["block"]["G"], "SDP eta*": sdp["block"]["eta_sdp"]})
block["1/sqrt(G)"] = 1 / np.sqrt(block["G"])
block["|difference|"] = (block["SDP eta*"] - block["1/sqrt(G)"]).abs()
display(block, pair)
print(f"largest deviation from the closed forms: {max(pair['|difference|'].max(), block['|difference|'].max()):.1e}")

,G,SDP eta*,1/sqrt(G),|difference|
0,1,1.000000,1.000000,4.074536e-09
1,2,0.707107,0.707107,1.553712e-09
2,3,0.577350,0.577350,1.208036e-09
3,4,0.500000,0.500000,2.460532e-12
4,5,0.447214,0.447214,3.100445e-10


,theta_deg,SDP eta*,closed form,|difference|
0,0.0,1.000000,1.000000,1.977072e-10
1,10.0,0.923062,0.923062,3.417449e-09
2,20.0,0.863218,0.863218,4.491317e-09
3,30.0,0.816497,0.816497,6.273341e-09
4,40.0,0.780206,0.780206,1.947367e-09
5,50.0,0.752487,0.752487,1.250140e-08
6,60.0,0.732051,0.732051,8.253077e-09
7,70.0,0.718015,0.718015,3.462976e-09
8,80.0,0.709808,0.709808,9.377998e-09
9,90.0,0.707107,0.707107,5.259175e-09


largest deviation from the closed forms: 1.3e-08


## Table 1

In [4]:
SHOW_THETAS = [0, 20, 40, 60, 90]
rows_b = block.copy(); rows_b["kappa"] = rows_b["G"]
rows_p = pair[pair["theta_deg"].round().isin(SHOW_THETAS)].reset_index(drop=True)
rows_p["kappa"] = 1 / rows_p["closed form"] ** 2            # = 1 + sin(theta)
table1 = pd.concat([rows_b[["G", "SDP eta*", "1/sqrt(G)", "kappa"]],
                    rows_p[["theta_deg", "SDP eta*", "closed form", "kappa"]]], axis=1)
display(table1.style.format(precision=5).format({("kappa"): "{:.3f}"}, subset=pd.IndexSlice[:, ["kappa"]]))

tex = []
for (_, b), (_, p) in zip(rows_b.iterrows(), rows_p.iterrows()):
    th = f"${p['theta_deg']:.0f}^\\circ$"
    tex.append(f"{b['G']:.0f} & {b['SDP eta*']:.5f} & {b['1/sqrt(G)']:.5f} & {b['kappa']:.0f} & "
               f"{th:<11s}& {p['SDP eta*']:.5f} & {p['closed form']:.5f} & {p['kappa']:.3f}\\\\")
(RESULTS / "table1_sdp.tex").write_text("\n".join(tex) + "\n")
print("\n".join(tex))

,G,SDP eta*,1/sqrt(G),kappa,theta_deg,SDP eta*,closed form,kappa
0,1,1.00000,1.00000,1.000,0.00000,1.00000,1.00000,1.000
1,2,0.70711,0.70711,2.000,20.00000,0.86322,0.86322,1.342
2,3,0.57735,0.57735,3.000,40.00000,0.78021,0.78021,1.643
3,4,0.50000,0.50000,4.000,60.00000,0.73205,0.73205,1.866
4,5,0.44721,0.44721,5.000,90.00000,0.70711,0.70711,2.000


1 & 1.00000 & 1.00000 & 1 & $0^\circ$  & 1.00000 & 1.00000 & 1.000\\
2 & 0.70711 & 0.70711 & 2 & $20^\circ$ & 0.86322 & 0.86322 & 1.342\\
3 & 0.57735 & 0.57735 & 3 & $40^\circ$ & 0.78021 & 0.78021 & 1.643\\
4 & 0.50000 & 0.50000 & 4 & $60^\circ$ & 0.73205 & 0.73205 & 1.866\\
5 & 0.44721 & 0.44721 & 5 & $90^\circ$ & 0.70711 & 0.70711 & 2.000\\


## Comparison with the data shipped with the paper

In [5]:
ref = json.loads((REFERENCE / "sdp_eta.json").read_text())
d = max(np.abs(np.subtract(ref["pair"]["eta_sdp"], sdp["pair"]["eta_sdp"])).max(),
        np.abs(np.subtract(ref["block"]["eta_sdp"], sdp["block"]["eta_sdp"])).max())
print(f"max |this run - reference/sdp_eta.json| = {d:.1e}")
assert d < 1e-6

max |this run - reference/sdp_eta.json| = 1.3e-13
